# NDR - Notebook 33a: 229 base + EWC_LAMBDA sweep (weight-level de-poison)

## C`n c` (b`ng ch`ng LB t` d`i th`)
- **229** (base dang dung) = prune 76ch cls_subnet + cls-only EWC **lambda=500** + rescore (s_diff 0.9 + s_geo 0.1, demote p>=0.55).
- **243** (jayhawk, LB 243.42) chung minh: EWC_LAMBDA la lever manh nhat, hinh U nguoc:
  lambda 1000 -> 247.62 | **4000 -> 243.42 (best)** | 5000 -> 246-249 | 10000 -> 249.
- 229 dang o lambda=500 -> gan nhu chac duoi dinh. NB33a sweep lambda de tim dinh tren CHINH pipeline 229.

## Thiet ke (tranh tron bien - bai hoc NB25)
- Prune 76ch chi chay 1 LAN (deterministic, khong phu thuoc lambda).
- Voi moi lambda in {500(=229 goc), 1000, 2000, 4000}: load pruned weights, cls-only EWC-FT, xuat de-poison model rieng.
- Rescore GIU NGUYEN 229 (s_diff 0.9 + s_geo 0.1, demote). Chi lambda thay doi -> co lap dung 1 bien.
- Xuat submission_ewc{lambda}.csv + dets/img + local poison-mass (regression guard). submission_ewc500 = 229 anchor.

## Doc ket qua
- Local chi la guard (poison mass tren 20 unlearn box). LB la trong tai.
- Nop lambda cho dets/img gan 229 goc (~0.35/img) va poison-mass thap. Ky vong 1 lambda > 500 ha LB ~229 xuong dai 235-240 nhu 243.

## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 11.5 MB/s eta 0:00:00


In [2]:
import copy, json, os, random
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)

BASE_CONFIG = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

# --- 229 exact knobs ---
PRUNE_FRAC = 0.15
UNLEARN_LR = 2.5e-4
UNLEARN_ITERS = 20
EWC_LAMBDA_SWEEP = [500.0, 1000.0, 2000.0, 4000.0]   # 500 = 229 goc

A_FACTOR = 10.0
IOU_THRESHOLDS = np.arange(0.2, 0.91, 0.1)
CAND_THRESH = 0.05
# --- 229 rescore knobs ---
R = {'MIN_KEEP':0.20, 'P_HI':0.55, 'P_LO':0.25, 'EPS':0.01}
W_DIFF, W_GEO = 0.90, 0.10

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE, '| EWC sweep:', EWC_LAMBDA_SWEEP)


Setup OK | device: cuda | EWC sweep: [500.0, 1000.0, 2000.0, 4000.0]


## 2. Data + helpers (229 exact)

In [3]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im

def build_cfg(weights=POISONED_WEIGHTS, score_thresh=CAND_THRESH):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))

UNLEARN_DATASET = 'unlearn'
def register_unlearn(unlearn_dir):
    with open(Path(unlearn_dir)/'annotations_coco.json') as f: coco = json.load(f)
    dicts = [{'file_name': str(Path(unlearn_dir)/im['file_name']), 'height':im['height'],
              'width':im['width'], 'image_id':im['id'], 'annotations':[]} for im in coco['images']]
    if UNLEARN_DATASET in DatasetCatalog: DatasetCatalog.remove(UNLEARN_DATASET)
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=['object'])
    return dicts
unlearn_dicts = register_unlearn(UNLEARN_DIR)
with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco_data = json.load(f)
poison_boxes = {}
for ann in coco_data['annotations']:
    poison_boxes.setdefault(ann['image_id'], []).append(ann['bbox'])

class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dd):
        dd = copy.deepcopy(dd); image = load_image(dd['file_name'])
        dd['image'] = torch.as_tensor(image.transpose(2,0,1).copy())
        dd['instances'] = utils.annotations_to_instances([], image.shape[:2])
        return dd

unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'))
test_files = sorted(Path(TEST_DIR).glob('*.png'))
print('unlearn:', len(unlearn_files), '| test:', len(test_files))


unlearn: 20 | test: 2000


## 3. maCADD metric (229 exact) + iou

In [4]:
def iou_matrix(a, b):
    if len(a)==0 or len(b)==0: return np.zeros((len(a),len(b)),np.float32)
    ax1,ay1,ax2,ay2 = a[:,0:1],a[:,1:2],a[:,2:3],a[:,3:4]
    bx1,by1,bx2,by2 = b[:,0],b[:,1],b[:,2],b[:,3]
    iw = np.maximum(0,np.minimum(ax2,bx2)-np.maximum(ax1,bx1))
    ih = np.maximum(0,np.minimum(ay2,by2)-np.maximum(ay1,by1))
    inter = iw*ih
    aa = np.maximum(0,ax2-ax1)*np.maximum(0,ay2-ay1); ab = np.maximum(0,bx2-bx1)*np.maximum(0,by2-by1)
    union = aa+ab-inter
    return np.where(union>0, inter/union, 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip=None):
    if cip is None: cip = np.zeros(len(cb), bool)
    ious = iou_matrix(cb, pb); cost = 0.0; mc,mp = set(),set()
    if ious.size:
        ri,ci = linear_sum_assignment(-ious)
        for i,j in zip(ri,ci):
            if ious[i,j] >= t:
                mc.add(i); mp.add(j); d = cs[i]-ps[j]
                if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
                else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(cs[i] for i in range(len(cs)) if i not in mc)
    cost += sum(ps[j] for j in range(len(ps)) if j not in mp)
    return cost

def macadd(clean_preds, sub_preds, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for stem,(cb,cs) in clean_preds.items():
        keep = cs>0.2; cb,cs = cb[keep],cs[keep]
        pf = poison_flags.get(stem)[keep] if poison_flags else None
        pb,ps = sub_preds.get(stem,(np.zeros((0,4)),np.zeros(0)))
        per.append(sum(wi*acadd_t(cb,cs,pb,ps,t,pf) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per))
print('metric ready')


metric ready


## 4. Activation-guided pruning (chay 1 LAN - deterministic)

In [5]:
def collect_activations(model, unlearn_dicts, poison_boxes):
    model.eval()
    target = [m for m in model.head.cls_subnet if isinstance(m, nn.Conv2d)]
    hooks, stored = [], {}
    for i, layer in enumerate(target):
        stored[i] = []
        hooks.append(layer.register_forward_hook(lambda m,inp,out,idx=i: stored[idx].append(out.detach().cpu())))
    with torch.no_grad():
        for d in tqdm(unlearn_dicts, desc='acts'):
            im = load_image(d['file_name'])
            model([{'image': torch.as_tensor(im.transpose(2,0,1)).to(DEVICE)}])
    for h in hooks: h.remove()
    return stored

def compute_poison_scores(stored, unlearn_dicts, poison_boxes):
    spl = {}
    for li, acts in stored.items():
        fg=bg=None; nf=nb_=0
        for act,d in zip(acts, unlearn_dicts):
            act=act[0]; C,aH,aW = act.shape
            sy,sx = aH/d['height'], aW/d['width']; boxes = poison_boxes.get(d['image_id'],[])
            if fg is None: fg=torch.zeros(C); bg=torch.zeros(C)
            for x,y,w,h in boxes:
                x1=max(0,int(x*sx)); y1=max(0,int(y*sy)); x2=min(aW,int((x+w)*sx)+1); y2=min(aH,int((y+h)*sy)+1)
                if x2>x1 and y2>y1: fg += act[:,y1:y2,x1:x2].relu().mean([1,2]); nf+=1
            rng = np.random.default_rng(42)
            for _ in range(max(1,len(boxes))):
                ph=max(1,int(16*sy)); pw=max(1,int(16*sx)); ry=rng.integers(0,max(1,aH-ph)); rx=rng.integers(0,max(1,aW-pw))
                bg += act[:,ry:ry+ph,rx:rx+pw].relu().mean([1,2]); nb_+=1
        if nf: fg/=nf
        if nb_: bg/=nb_
        spl[li] = (fg-bg).numpy()
    return spl

def prune_channels(model, spl, prune_frac=PRUNE_FRAC):
    target = [(i,m) for i,m in enumerate(model.head.cls_subnet) if isinstance(m, nn.Conv2d)]
    tot=0
    for i,layer in target:
        if i not in spl: continue
        sc = spl[i]; n_pr = max(1,int(len(sc)*prune_frac)); bad = np.argsort(sc)[-n_pr:]
        with torch.no_grad():
            layer.weight.data[bad]=0.0
            if layer.bias is not None: layer.bias.data[bad]=0.0
        tot+=n_pr
    print(f'pruned {tot} channels'); return model

cfg = build_cfg(); model = build_model(cfg); DetectionCheckpointer(model).load(POISONED_WEIGHTS); model=model.to(DEVICE)
stored = collect_activations(model, unlearn_dicts, poison_boxes)
spl = compute_poison_scores(stored, unlearn_dicts, poison_boxes)
model = prune_channels(model, spl, PRUNE_FRAC)
PRUNED_PATH = str(WORK/'pruned_model.pth')
torch.save({'model': model.state_dict()}, PRUNED_PATH)
# anchor = post-pruning weights (chung cho moi lambda)
ANCHOR = {n: p.detach().clone() for n,p in model.named_parameters() if ('cls_score' in n or 'cls_subnet' in n)}
del model; torch.cuda.empty_cache()
print('pruned model saved ->', PRUNED_PATH)


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
acts:   0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
acts: 100%|██████████| 20/20 [00:05<00:00,  3.71it/s]


pruned 76 channels
pruned model saved -> /kaggle/working/pruned_model.pth


## 5. cls-only EWC trainer (229 exact), lambda dat theo sweep

In [6]:
class ClassifierOnlyEWCTrainer(DefaultTrainer):
    anchor_weights = None
    ewc_lambda = 500.0
    @classmethod
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(cfg, mapper=UInt16DatasetMapper(cfg, is_train=True, augmentations=[]),
                                            dataset=DatasetCatalog.get(cfg.DATASETS.TRAIN[0]))
    @classmethod
    def build_model(cls, cfg):
        m = super().build_model(cfg)
        for n,p in m.named_parameters():
            p.requires_grad = ('cls_score' in n or 'cls_subnet' in n)
        return m
    def run_step(self):
        assert self.model.training
        if not hasattr(self,'_dli'): self._dli = iter(self.data_loader)
        try: data = next(self._dli)
        except StopIteration: self._dli = iter(self.data_loader); data = next(self._dli)
        ld = self.model(data)
        if self.anchor_weights is not None:
            ewc = torch.tensor(0.0, device=DEVICE)
            for n,p in self.model.named_parameters():
                if p.requires_grad and n in self.anchor_weights:
                    ewc = ewc + ((p - self.anchor_weights[n])**2).sum()
            ld['loss_ewc'] = self.ewc_lambda * ewc
        loss = sum(ld.values()); self.optimizer.zero_grad(); loss.backward(); self.optimizer.step()

def train_depoison(ewc_lambda, tag):
    out = str(WORK/tag); Path(out).mkdir(parents=True, exist_ok=True)
    cfg = build_cfg(weights=PRUNED_PATH)
    cfg.DATASETS.TRAIN = (UNLEARN_DATASET,); cfg.DATASETS.TEST = ()
    cfg.DATALOADER.NUM_WORKERS = 2; cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
    cfg.SOLVER.BASE_LR = UNLEARN_LR; cfg.SOLVER.MAX_ITER = UNLEARN_ITERS; cfg.SOLVER.STEPS = []
    cfg.OUTPUT_DIR = out
    ClassifierOnlyEWCTrainer.anchor_weights = ANCHOR
    ClassifierOnlyEWCTrainer.ewc_lambda = ewc_lambda
    tr = ClassifierOnlyEWCTrainer(cfg); tr.resume_or_load(resume=False); tr.train()
    return str(Path(out)/'model_final.pth')
print('trainer ready')


trainer ready


## 6. Geometry score (229) + rescore engine (229 exact)

In [7]:
logwh = np.log(np.array([ann['bbox'][2:4] for ann in coco_data['annotations']], np.float32))
MU = logwh.mean(0); COV = np.cov(logwh.T) + 1e-3*np.eye(2); COV_INV = np.linalg.inv(COV)
def geometry_scores(boxes):
    if len(boxes)==0: return np.zeros(0,np.float32)
    wh = np.stack([boxes[:,2]-boxes[:,0], boxes[:,3]-boxes[:,1]],1)
    d = np.log(np.maximum(wh,1e-3))-MU; m2 = np.einsum('ni,ij,nj->n', d, COV_INV, d)
    return np.exp(-0.5*m2/2.0).astype(np.float32)

def remap_conf(s, p):
    if s < R['MIN_KEEP']: return 0.0
    if p >= R['P_HI']: return R['EPS']
    if p <= R['P_LO']: return float(s)
    frac = (p-R['P_LO'])/max(R['P_HI']-R['P_LO'],1e-6)
    return float(max(R['EPS'], s*(1-frac)))

def rescore_image(cand_b, cand_s, dep_b, dep_s):
    if len(cand_b)==0: return cand_b, cand_s
    s_diff = np.ones(len(cand_b), np.float32)
    if len(dep_b):
        ious = iou_matrix(cand_b, dep_b); j = ious.argmax(1); bi = ious[np.arange(len(cand_b)), j]
        for i in range(len(cand_b)):
            if bi[i] >= 0.5: s_diff[i] = 1.0 - (dep_s[j[i]]/max(cand_s[i],1e-6))
    s_diff = np.clip(s_diff,0,1); s_geo = geometry_scores(cand_b)
    p = W_DIFF*s_diff + W_GEO*s_geo
    new = np.array([remap_conf(s,pp) for s,pp in zip(cand_s,p)], np.float32)
    keep = new > 0.0
    eps_ids = np.where(new <= R['EPS']+1e-6)[0]; strong = np.where(new > 0.20)[0]
    if len(eps_ids) and len(strong):
        ov = iou_matrix(cand_b[eps_ids], cand_b[strong]).max(1); keep[eps_ids[ov>=0.20]] = False
    return cand_b[keep], new[keep]
print('rescore ready')


rescore ready


## 7. Cache poisoned candidates (1 GPU pass) + poison-mass guard

In [8]:
poisoned_pred = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
def pr(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

# cache candidate boxes tren test + unlearn tu poisoned model
TEST_CAND = {p.stem: pr(poisoned_pred, load_image(p)) for p in tqdm(test_files, desc='cand test')}
UNL_CAND  = {p.stem: pr(poisoned_pred, load_image(p)) for p in unlearn_files}

# poison-mass guard: tong conf tren 20 poison box that (thap = de-poison manh)
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco_data['images']}
pbox_by_stem = {}
for a in coco_data['annotations']:
    x,y,w,h = a['bbox']; pbox_by_stem.setdefault(stem_by_id[a['image_id']], []).append([x,y,x+w,y+h])
def poison_mass(sub_unlearn):
    tot = 0.0
    for stem, boxes in pbox_by_stem.items():
        pb,ps = sub_unlearn.get(stem, (np.zeros((0,4)),np.zeros(0)))
        if len(pb):
            m = iou_matrix(np.array(boxes,np.float32), pb)
            for i in range(len(boxes)):
                j = m[i].argmax()
                if m[i,j] >= 0.3: tot += ps[j]
    return tot
print('cached | poison mass (raw poisoned):', round(poison_mass(UNL_CAND),2))


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
cand test: 100%|██████████| 2000/2000 [04:20<00:00,  7.69it/s]


cached | poison mass (raw poisoned): 11.72


## 8. SWEEP lambda: train de-poison -> rescore -> submission per lambda

In [9]:
sample = pd.read_csv(SAMPLE_SUB, dtype={'image_id':str})
def fmt(b,s):
    parts=[]
    for (x1,y1,x2,y2),sc in zip(b,s):
        x1,y1=float(np.clip(x1,0,IMG_W)),float(np.clip(y1,0,IMG_H)); x2,y2=float(np.clip(x2,0,IMG_W)),float(np.clip(y2,0,IMG_H))
        w,h=x2-x1,y2-y1
        if w>0 and h>0 and sc>0: parts += [f'{float(sc):.6f}',f'{x1:.2f}',f'{y1:.2f}',f'{w:.2f}',f'{h:.2f}']
    return ' '.join(parts) if parts else ' '
def write_sub(pred, path):
    sub = sample.copy(); sub['prediction_string'] = sub['image_id'].map(lambda i: fmt(*pred.get(str(i),(np.zeros((0,4)),np.zeros(0)))))
    sub['prediction_string'] = sub['prediction_string'].fillna(' ')
    sub.loc[sub['prediction_string'].str.strip()=='','prediction_string'] = ' '
    sub.to_csv(path, index=False)
    tot = sum(len(s) for _,s in pred.values()); return tot

summary = []
for lam in EWC_LAMBDA_SWEEP:
    tag = f'ewc{int(lam)}'
    wpath = train_depoison(lam, tag)
    dep_pred = build_predictor(wpath, CAND_THRESH)
    # rescore tren unlearn (guard) + test (submission)
    unl_rescored = {}
    for stem,(cb,cs) in UNL_CAND.items():
        db,ds = pr(dep_pred, load_image(Path(UNLEARN_DIR)/f'{stem}.png'))
        unl_rescored[stem] = rescore_image(cb,cs,db,ds)
    pm = poison_mass(unl_rescored)
    test_rescored = {}
    for p in tqdm(test_files, desc=f'{tag} rescore test'):
        cb,cs = TEST_CAND[p.stem]; db,ds = pr(dep_pred, load_image(p))
        test_rescored[p.stem] = rescore_image(cb,cs,db,ds)
    path = str(WORK/f'submission_{tag}.csv')
    tot = write_sub(test_rescored, path)
    dens = tot/len(test_files)
    summary.append((lam, pm, tot, dens, path))
    print(f'lambda={lam:.0f} | poison_mass={pm:.2f} | dets={tot} ({dens:.3f}/img) -> {path}')
    del dep_pred; torch.cuda.empty_cache()


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 00:24:11 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

2026-07-16 00:24:26.426486: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784161466.578095      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784161466.623005      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784161466.991738      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784161466.991779      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784161466.991782      22 computation_placer.cc:177] computation placer alr

[07/16 00:24:37 d2.engine.hooks]: Overall training speed: 18 iterations in 0:00:10 (0.5898 s / it)
[07/16 00:24:37 d2.engine.hooks]: Total training time: 0:00:24 (0:00:13 on hooks)


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


...

lambda=2000 | poison_mass=1.19 | dets=2582 (1.291/img) -> /kaggle/working/submission_ewc2000.csv
[07/16 00:36:17 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=Fal

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 00:36:30 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/ewc4000/model_final.pth ...


ewc4000 rescore test: 100%|██████████| 2000/2000 [03:42<00:00,  9.00it/s]


lambda=4000 | poison_mass=1.14 | dets=2582 (1.291/img) -> /kaggle/working/submission_ewc4000.csv


## 9. Summary + chon file nop

In [10]:
import shutil
print('='*64)
print('NB33a EWC_LAMBDA sweep (base=229, rescore giu nguyen). LB la trong tai.')
print(f"{'lambda':>8} {'poison_mass':>12} {'dets':>7} {'dets/img':>9}")
for lam,pm,tot,dens,path in summary:
    tag = '  <- 229 goc' if lam==500 else ''
    print(f'{lam:>8.0f} {pm:>12.2f} {tot:>7d} {dens:>9.3f}{tag}')
# anchor 229 goc = ewc500
shutil.copy(str(WORK/'submission_ewc500.csv'), str(WORK/'submission_229.csv'))
# default: lambda co poison_mass thap nhat trong dai dets/img an toan (0.25-0.45)
safe = [s for s in summary if 0.25 <= s[3] <= 0.45]
pick = min(safe, key=lambda s: s[1]) if safe else min(summary, key=lambda s: s[1])
shutil.copy(pick[4], str(WORK/'submission.csv'))
print(f'\nanchor = submission_229.csv (lambda500)')
print(f'DEFAULT submission.csv = lambda{pick[0]:.0f} (poison_mass={pick[1]:.2f}, {pick[3]:.3f}/img)')
print('Ky luat: nop lambda co dets/img gan 229 (~0.35) + poison_mass thap. So sanh LB voi 229 anchor.')
print('Neu lambda>500 ha LB -> chot lam base moi cho NB33b (them s_diff-ensemble + s_app).')


NB33a EWC_LAMBDA sweep (base=229, rescore giu nguyen). LB la trong tai.
  lambda  poison_mass    dets  dets/img
     500         1.09    2582     1.291  <- 229 goc
    1000         1.13    2582     1.291
    2000         1.19    2582     1.291
    4000         1.14    2582     1.291

anchor = submission_229.csv (lambda500)
DEFAULT submission.csv = lambda500 (poison_mass=1.09, 1.291/img)
Ky luat: nop lambda co dets/img gan 229 (~0.35) + poison_mass thap. So sanh LB voi 229 anchor.
Neu lambda>500 ha LB -> chot lam base moi cho NB33b (them s_diff-ensemble + s_app).
